<div dir="rtl" style="text-align:right">
<h1>یک شبکهٔ کوچک واقعاً چه چیزی یاد می‌گیرد؟</h1><p style="text-align:right"><b>پرسش آزمایش:</b> چرا Activation می‌تواند نتیجهٔ یادگیری XOR را تغییر دهد؟</p><p style="text-align:right">پیش‌نیاز: <a href="http://127.0.0.1:8000/part-03/chapter-03/18-module.html"><bdi dir="ltr">18-module</bdi></a>، <a href="http://127.0.0.1:8000/part-03/chapter-03/19-network.html"><bdi dir="ltr">19-network</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای اجرای دوباره از ابتدا، Kernel را Restart و سپس Run All کنید. برای بازکردن لینک درس‌ها، سرور کتاب باید روی پورت ۸۰۰۰ اجرا شده باشد؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)

import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(17)

def inspect(name, value):
    print(name, "shape =", tuple(value.shape),
          "dtype =", value.dtype, "device =", value.device)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">چهار ورودی XOR را خودمان می‌سازیم؛ نمونهٔ Test مستقلی نداریم. موفقیت روی این چهار نقطه فقط یادگیری همین جدول را نشان می‌دهد. پیش از اجرا تعداد پارامترهای شبکهٔ ۲→۸→۲ با Bias را حساب کنید.</p>
</div>

In [ ]:
from torch import nn
from torch.nn import functional as F
x = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
targets = torch.tensor([0,1,1,0], dtype=torch.long)
inspect("features", x)
inspect("class IDs", targets)

def train_network(nonlinear, width=8, seed=9):
    torch.manual_seed(seed)
    network = nn.Sequential(nn.Linear(2,width),
                            nn.Tanh() if nonlinear else nn.Identity(),
                            nn.Linear(width,2))
    optimizer = torch.optim.Adam(network.parameters(), lr=0.05)
    history = []
    for _ in range(400):
        optimizer.zero_grad(set_to_none=True)
        logits = network(x)
        loss = F.cross_entropy(logits, targets)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    return network, history

network, history = train_network(True)
assert sum(p.numel() for p in network.parameters()) == 42
inspect("logits", network(x))
print("Predictions:", network(x).argmax(-1).tolist())
assert network(x).argmax(-1).tolist() == targets.tolist()


<div dir="rtl" style="text-align:right">
<h2>فقط Activation را حذف کنیم</h2><p style="text-align:right">شبکه و Optimizer را از نو می‌سازیم و همان Seed را می‌گذاریم. <code dir="ltr" style="unicode-bidi:isolate">nn.Identity()</code> ورودی را بی‌تغییر عبور می‌دهد و اینجا جای Activation را می‌گیرد. دو تبدیل Affine بدون Activation در یک تبدیل Affine خلاصه می‌شوند. پیش‌بینی کنید چرا جداسازی XOR برای این مسیر ممکن نیست.</p>
</div>

In [ ]:
linear, linear_history = train_network(False)
print("Without activation:", linear(x).argmax(-1).tolist())
fig, ax = plt.subplots(figsize=(6,3))
ax.plot(history, label="Tanh")
ax.plot(linear_history, label="No activation")
ax.set(xlabel="Step", ylabel="Training Loss")
ax.legend()
plt.show()
try:
    F.cross_entropy(network(x), targets.float())
except RuntimeError as error:
    print("Expected class-ID dtype failure:", error)
else:
    raise AssertionError("Integer class IDs are required here")


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>تمرین:</b> اجرای مرجع بالا را نگه دارید. در Cell تازه، تابع <code dir="ltr" style="unicode-bidi:isolate">train_network(True, width=2, seed=4)</code> را فراخوانی کنید و مدل و تاریخچهٔ تازه را بگیرید؛ سپس تعداد پارامتر و دقت آن را گزارش کنید، بی‌آنکه موفقیت کامل را assert کنید. چند Seed را جدا بسنجید. موفقیت یا شکست یک اجرای کوتاه را با اثبات ظرفیت یا تعمیم یکی نگیرید. در مدل زبان، همین تفکیکِ Forward، Loss، Gradient و step را با داده‌ای بزرگ‌تر دنبال خواهیم کرد.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2>برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی و نتیجهٔ اجرا را کنار هم بنویسید؛ اگر تفاوتی داشتند، دلیلش را توضیح دهید. سپس به <a href="http://127.0.0.1:8000/part-03/chapter-03/19-network.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>